# Master's Thesis Run Notebook

This notebook is part of the **Master's Thesis (MSc Dissertation)**: **Fast Simulation of Neutrino Oscillations in Matter**.

**Author**  
Juan Ramon Diaz Santos <diazjuan@alumni.uv.es>

**Supervisors**  
Roberto Ruiz de Austri Bazan <rruiz@ific.uv.es>  
Michele Lucente <michele.lucente@unibo.it>

**Date**  
June 2026


# Solar 1: Solar-to-Detector Propagation

Run solar-neutrino propagation from production in the Sun to detector-level probabilities. The notebook can dispatch the coherent, incoherent, legacy, or all configured implementations and saves one Torch file per selected mode.


## 1. Libraries

This section imports Python standard-library modules, PyTorch, repository helpers, and the TPeanuts APIs required by the solar propagation run. It also locates the repository root so the notebook can be executed from Jupyter without relying on the script `__file__` variable.


In [1]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path
from typing import Literal

import torch

from tpeanuts.flux_propagation import (
    run_and_save_solar_to_detector_coherent,
    run_and_save_solar_to_detector_incoherent,
    run_and_save_solar_to_detector_legacypeanuts,
)
from tpeanuts.util.torch_util import _default_device, resolve_device

from tpeanuts.util.notebooks import find_repo_root
HERE = Path.cwd().resolve()
PACKAGE_DIR = find_repo_root(HERE, folder="analysis")
print(f"PACKAGE_DIR = {PACKAGE_DIR}")


PACKAGE_DIR = G:\Mi unidad\03.Codigo\034.TFM.UV\Tpeanuts


## 2. Paths and Configuration

This section centralizes every filesystem path and all run parameters before any computation is started. Keeping these values together makes the run reproducible and avoids hidden output locations.


### 2.1. Paths

All outputs are rooted at `DEFAULT_OUTPUT_ROOT` unless the environment variable `TPEANUTS_OUTPUT_ROOT` is set. Derived folders are created from that single root, with solar detector outputs written under `OUTPUT_DATA_ROOT / "solar" / "detector"`.


In [2]:
DEFAULT_OUTPUT_ROOT = Path(r"V:\output")
OUTPUT_ROOT = Path(os.environ.get("TPEANUTS_OUTPUT_ROOT", DEFAULT_OUTPUT_ROOT))

OUTPUT_DATA_ROOT = Path(OUTPUT_ROOT / "data")
OUTPUT_ANALYSIS_ROOT = Path(OUTPUT_ROOT / "analysis")
OUTPUT_BENCHMARK_ROOT = Path(OUTPUT_ROOT / "benchmark")
OUTPUT_TEST_ROOT = Path(OUTPUT_ROOT / "test")

OUTPUT_DATA_ATMOSPHERE = Path(OUTPUT_DATA_ROOT / "atmosphere")
OUTPUT_DATA_SOLAR = Path(OUTPUT_DATA_ROOT / "solar")
OUTPUT_DATA_EXTERNAL = Path(OUTPUT_DATA_ROOT / "external")

OUTPUT_ANALYSIS_ATMOSPHERE = Path(OUTPUT_ANALYSIS_ROOT / "atmosphere")
OUTPUT_ANALYSIS_SOLAR = Path(OUTPUT_ANALYSIS_ROOT / "solar")
OUTPUT_ANALYSIS_EXTERNAL = Path(OUTPUT_ANALYSIS_ROOT / "external")

OUTPUT_DATA_MCEQ = Path(OUTPUT_DATA_EXTERNAL / "mceq")
OUTPUT_DATA_HONDA = Path(OUTPUT_DATA_EXTERNAL / "honda")

OUTPUT_SOLARDETECTOR_ROOT = Path(OUTPUT_DATA_SOLAR / "detector")
OUTPUT_FILENAME = "solardetector.pt"
OUTPUT_SOLARDETECTOR_ROOT.mkdir(parents=True, exist_ok=True)


## 2.2. Parameters and System Configuration

This subsection defines the system configuration used by the run. The parameters control the selected propagation implementation, oscillation physics, solar source, energy grid, detector geometry, exposure integration, runtime device, and output precision. Adjust these values before executing the final production cell. Possible problems include missing legacy input files, unavailable GPU devices, long exposure integrations, or accidental overwriting when `OVERWRITE=True`.


### Simulation selector

These parameters choose which solar propagation implementation will run. Use `coherent`, `incoherent`, or `legacy` for a single production, or `ALL` to execute every configured mode.


In [3]:
SimulationMode = Literal["coherent", "incoherent", "legacy", 'ALL']

SIMULATION_MODES = ["coherent", "incoherent", "legacy"]
SIMULATION_MODE: SimulationMode = "ALL"


### Data products

These parameters define the saved solar-detector files, including the base filename, overwrite policy, and tensor precision used on disk.


In [4]:
OVERWRITE = True
SAVE_DTYPE = torch.float32


### Oscillation physics

These parameters define the mass splittings, mixing angles, CP phase, and neutrino/antineutrino branch used by all propagation modes.


In [5]:
DM21_EV2 = 7.42e-5
DM3L_EV2 = 2.517e-3

THETA12 = 0.59
THETA13 = 0.15
THETA23 = 0.78

DELTA_CP = 1.20

ANTINU = False


### Solar production

These parameters define the solar source and production model. The legacy file paths are explicit so the notebook does not rely on hidden legacy defaults.


In [6]:
SOURCE = "8B"

coherent_PRODUCTION_MODE = "incoherent"

RHO0 = 0.08

INITIAL_STATE = "nue"

LEGACY_solar_MODEL_FILE = str(PACKAGE_DIR / "data" / "peanuts" / "nudistr_b16_agss09.dat")
LEGACY_solar_flux_FILE = str(PACKAGE_DIR / "data" / "peanuts" / "fluxes_b16.dat")


### Energy grid

These parameters define the neutrino energy grid in MeV. Increasing the number of points improves resolution but increases runtime.


In [7]:
E_MIN_MEV = 1.0
E_MAX_MEV = 15.0
E_N = 21

E_GRID_MEV = torch.linspace(E_MIN_MEV, E_MAX_MEV, E_N, dtype=torch.float64)


### Earth and detector

These parameters define the Earth-density input, detector depth and latitude, and whether Earth propagation is reunitarized.


In [8]:
EARTH_DENSITY_FILE = str(PACKAGE_DIR / "data" / "density" / "earth_density.csv")
LEGACY_EARTH_DENSITY_FILE = str(PACKAGE_DIR / "data" / "peanuts" / "earth_density.csv")

TABULATED_earth_density = False

DETECTOR_DEPTH_M = 1000.0

DETECTOR_LATITUDE_RAD = 0.72

REUNITARIZE_earth = False


### Sun-Earth distance

These parameters control whether the Sun-Earth distance is read from the tabulated data or supplied manually.


In [9]:
USE_SUN_EARTH_DISTANCE_TABLE = True

SUN_EARTH_DISTANCE_PATH = str(PACKAGE_DIR / "data" / "solar" / "sun_earth_distance.csv")

earth_DISTANCE_KM = None


### Nadir exposure

These parameters define the exposure integration grid and cache behavior used to integrate detector probabilities over nadir angle.


In [10]:
EXPOSURE_SOURCE = "math"

EXPOSURE_CSV_PATH = None

EXPOSURE_ANGLE = "Nadir"

EXPOSURE_DAYNIGHT = None
EXPOSURE_D1 = 0.0
EXPOSURE_D2 = 365.0
EXPOSURE_NS = 1000

EXPOSURE_CACHE_DIR = str(PACKAGE_DIR / "data" / "cache" / "cache_exposure")
EXPOSURE_USE_CACHE = False

LEGACY_EXPOSURE_NORMALIZED = True
LEGACY_EXPOSURE_FROM_FILE = None

ETA_GRID = None

INTEGRATE_EXPOSURE = True


### Runtime

These parameters control the compute device, numerical dtype, and verbosity used by the selected propagation mode.


In [11]:
DEVICE = "cuda:0"
COMPUTE_DTYPE = torch.float64

DEBUG = True


## 3. Helper Functions

The following cells define the helper functions used by the run. Each function is kept in its own code cell so it can be inspected, edited, and rerun independently during interactive validation.


### Function: `output_dir_for_mode`

This helper prepares arguments, resolves devices, formats output locations, or dispatches one selected propagation mode. The expected result is a reusable function that either returns a configuration dictionary or saves one configured solar-detector result.


In [12]:
def output_dir_for_mode(mode: SimulationMode) -> str:
    return str(Path(OUTPUT_SOLARDETECTOR_ROOT) / mode)


### Function: `filename_for_mode`

This helper prepares arguments, resolves devices, formats output locations, or dispatches one selected propagation mode. The expected result is a reusable function that either returns a configuration dictionary or saves one configured solar-detector result.


In [13]:
def filename_for_mode(mode: SimulationMode) -> str:
    stem = Path(OUTPUT_FILENAME).stem
    suffix = Path(OUTPUT_FILENAME).suffix or ".pt"
    return f"{stem}_{mode}{suffix}"


### Function: `common_kwargs`

This helper prepares arguments, resolves devices, formats output locations, or dispatches one selected propagation mode. The expected result is a reusable function that either returns a configuration dictionary or saves one configured solar-detector result.


In [14]:
def common_kwargs() -> dict:
    return {
        "E_MeV": E_GRID_MEV,
        "DeltamSq21": DM21_EV2,
        "DeltamSq3l": DM3L_EV2,
        "theta12": THETA12,
        "theta13": THETA13,
        "theta23": THETA23,
        "delta": DELTA_CP,
        "earth_distance_km": earth_DISTANCE_KM,
        "sun_earth_distance_path": SUN_EARTH_DISTANCE_PATH,
        "use_sun_earth_distance_table": USE_SUN_EARTH_DISTANCE_TABLE,
        "eta": ETA_GRID,
        "detector_depth_m": DETECTOR_DEPTH_M,
        "detector_latitude_rad": DETECTOR_LATITUDE_RAD,
        "exposure_d1": EXPOSURE_D1,
        "exposure_d2": EXPOSURE_D2,
        "exposure_ns": EXPOSURE_NS,
        "exposure_daynight": EXPOSURE_DAYNIGHT,
        "integrate_exposure": INTEGRATE_EXPOSURE,
        "antinu": ANTINU,
        "device": DEVICE,
        "dtype": COMPUTE_DTYPE,
        "debug": DEBUG,
    }


### Function: `coherent_kwargs`

This helper prepares arguments, resolves devices, formats output locations, or dispatches one selected propagation mode. The expected result is a reusable function that either returns a configuration dictionary or saves one configured solar-detector result.


In [15]:
def coherent_kwargs() -> dict:
    kwargs = common_kwargs()
    kwargs.update(
        {
            "initial_state": INITIAL_STATE,
            "production_mode": coherent_PRODUCTION_MODE,
            "rho0": RHO0,
            "source": SOURCE if coherent_PRODUCTION_MODE != "point" else None,
            "earth_density_file": EARTH_DENSITY_FILE,
            "tabulated_earth_density": TABULATED_earth_density,
            "exposure_source": EXPOSURE_SOURCE,
            "exposure_csv_path": EXPOSURE_CSV_PATH,
            "exposure_angle": EXPOSURE_ANGLE,
            "exposure_cache_dir": EXPOSURE_CACHE_DIR,
            "exposure_use_cache": EXPOSURE_USE_CACHE,
            "reunitarize_earth": REUNITARIZE_earth,
        }
    )
    return kwargs


### Function: `incoherent_kwargs`

This helper prepares arguments, resolves devices, formats output locations, or dispatches one selected propagation mode. The expected result is a reusable function that either returns a configuration dictionary or saves one configured solar-detector result.


In [16]:
def incoherent_kwargs() -> dict:
    kwargs = common_kwargs()
    kwargs.update(
        {
            "source": SOURCE,
            "earth_density_file": EARTH_DENSITY_FILE,
            "tabulated_earth_density": TABULATED_earth_density,
            "exposure_source": EXPOSURE_SOURCE,
            "exposure_csv_path": EXPOSURE_CSV_PATH,
            "exposure_angle": EXPOSURE_ANGLE,
            "exposure_cache_dir": EXPOSURE_CACHE_DIR,
            "exposure_use_cache": EXPOSURE_USE_CACHE,
            "reunitarize_earth": REUNITARIZE_earth,
        }
    )
    return kwargs


### Function: `legacy_kwargs`

This helper prepares arguments, resolves devices, formats output locations, or dispatches one selected propagation mode. The expected result is a reusable function that either returns a configuration dictionary or saves one configured solar-detector result.


In [17]:
def legacy_kwargs() -> dict:
    kwargs = common_kwargs()
    kwargs.update(
        {
            "source": SOURCE,
            "solar_model_file": LEGACY_solar_MODEL_FILE,
            "solar_flux_file": LEGACY_solar_flux_FILE,
            "earth_density_file": LEGACY_EARTH_DENSITY_FILE,
            "tabulated_earth_density": TABULATED_earth_density,
            "exposure_normalized": LEGACY_EXPOSURE_NORMALIZED,
            "exposure_from_file": LEGACY_EXPOSURE_FROM_FILE,
            "exposure_angle": EXPOSURE_ANGLE,
        }
    )
    return kwargs


## 4.Main run function: `run_selected_mode`

This helper prepares arguments, resolves devices, formats output locations, or dispatches one selected propagation mode. The expected result is a reusable function that either returns a configuration dictionary or saves one configured solar-detector result.


In [18]:
def run_selected_mode(mode: SimulationMode):
    mode = mode.lower()
    if mode not in ("coherent", "incoherent", "legacy"):
        raise ValueError("SIMULATION_MODE must be 'coherent', 'incoherent' or 'legacy'.")

    output_dir = output_dir_for_mode(mode)
    filename = filename_for_mode(mode)

    print("\n" + "=" * 80)
    print(f"RUN 3 solar DETECTOR PROPAGATION | mode={mode}")
    print("=" * 80)
    print(f"E grid       : {E_GRID_MEV.numel()} points [{E_GRID_MEV[0].item():.3g}, {E_GRID_MEV[-1].item():.3g}] MeV")
    print(f"source       : {SOURCE}")
    print(f"output       : {Path(output_dir) / filename}")
    print(f"device       : {resolve_device(DEVICE)}")
    print(f"exposure ns  : {EXPOSURE_NS}")

    t0 = time.perf_counter()

    if mode == "coherent":
        result = run_and_save_solar_to_detector_coherent(
            output_dir,
            filename=filename,
            overwrite=OVERWRITE,
            save_dtype=SAVE_DTYPE,
            **coherent_kwargs(),
        )
    elif mode == "incoherent":
        result = run_and_save_solar_to_detector_incoherent(
            output_dir,
            filename=filename,
            overwrite=OVERWRITE,
            save_dtype=SAVE_DTYPE,
            **incoherent_kwargs(),
        )
    else:
        result = run_and_save_solar_to_detector_legacypeanuts(
            output_dir,
            filename=filename,
            overwrite=OVERWRITE,
            save_dtype=SAVE_DTYPE,
            **legacy_kwargs(),
        )

    elapsed = time.perf_counter() - t0

    print("\nFinished.")
    print(f"elapsed      : {elapsed:.2f} s")
    print(f"saved        : {result['output_path']}")

    p_int = result.get("detector_probabilities_integrated")
    if torch.is_tensor(p_int):
        print(f"Pdet int     : shape={tuple(p_int.shape)}")
        print(f"Pdet sum min : {torch.min(p_int.sum(dim=-1)).item():.6g}")
        print(f"Pdet sum max : {torch.max(p_int.sum(dim=-1)).item():.6g}")

    return result


## 5. Execute Run

The final cell launches the configured production run and stores outputs in `RUN_RESULTS`. When `SIMULATION_MODE` is `ALL`, every mode listed in `SIMULATION_MODES` is executed in sequence; otherwise only the selected mode is run.


In [19]:
RUN_RESULTS = {}

if SIMULATION_MODE == "ALL":
    for mode in SIMULATION_MODES:
        RUN_RESULTS[mode] = run_selected_mode(mode)
else:
    RUN_RESULTS[SIMULATION_MODE] = run_selected_mode(SIMULATION_MODE)

RUN_RESULTS



RUN 3 solar DETECTOR PROPAGATION | mode=coherent
E grid       : 21 points [1, 15] MeV
source       : 8B
output       : V:\output\data\solar\detector\coherent\solardetector_coherent.pt
device       : cuda:0
exposure ns  : 1000
coherent solar detector pipeline: n_E=21, n_rho=1000, n_eta=1000, production_mode=incoherent

Finished.
elapsed      : 123.53 s
saved        : V:\output\data\solar\detector\coherent\solardetector_coherent.pt
Pdet int     : shape=(21, 3)
Pdet sum min : 1
Pdet sum max : 1.00001

RUN 3 solar DETECTOR PROPAGATION | mode=incoherent
E grid       : 21 points [1, 15] MeV
source       : 8B
output       : V:\output\data\solar\detector\incoherent\solardetector_incoherent.pt
device       : cuda:0
exposure ns  : 1000
Incoherent solar detector pipeline: n_E=21, n_eta=1000, source=8B

Finished.
elapsed      : 216.25 s
saved        : V:\output\data\solar\detector\incoherent\solardetector_incoherent.pt
Pdet int     : shape=(21, 3)
Pdet sum min : 1
Pdet sum max : 1.00002

RUN 3 so

{'coherent': {'mode': 'fully_coherent_solar_to_detector',
  'production_mode': 'incoherent',
  'flavour_order': ['nue', 'numu', 'nutau'],
  'E_MeV': tensor([ 1.0000,  1.7000,  2.4000,  3.1000,  3.8000,  4.5000,  5.2000,  5.9000,
           6.6000,  7.3000,  8.0000,  8.7000,  9.4000, 10.1000, 10.8000, 11.5000,
          12.2000, 12.9000, 13.6000, 14.3000, 15.0000], device='cuda:0',
         dtype=torch.float64),
  'rho_grid': tensor([0.0005, 0.0010, 0.0015, 0.0020, 0.0025, 0.0030, 0.0035, 0.0040, 0.0045,
          0.0050, 0.0055, 0.0060, 0.0065, 0.0070, 0.0075, 0.0080, 0.0085, 0.0090,
          0.0095, 0.0100, 0.0105, 0.0110, 0.0115, 0.0120, 0.0125, 0.0130, 0.0135,
          0.0140, 0.0145, 0.0150, 0.0155, 0.0160, 0.0165, 0.0170, 0.0175, 0.0180,
          0.0185, 0.0190, 0.0195, 0.0200, 0.0205, 0.0210, 0.0215, 0.0220, 0.0225,
          0.0230, 0.0235, 0.0240, 0.0245, 0.0250, 0.0255, 0.0260, 0.0265, 0.0270,
          0.0275, 0.0280, 0.0285, 0.0290, 0.0295, 0.0300, 0.0305, 0.0310, 0.0315,